<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 6 章练习解答

## 练习 6.1：增加上下文长度

**练习目的**：将输入填充到模型支持的最大 token 数，观察对分类性能的影响。

**关键洞察**：更长的上下文并不一定带来更好的结果。对于短文本分类任务，大量填充可能引入噪声，反而降低性能。

我们可以通过将 max length 设为 1024 来将输入填充到模型支持的最大 token 数：

```python
max_length = 1024

train_dataset = SpamDataset(base_path / "train.csv", max_length=max_length, tokenizer=tokenizer)
val_dataset = SpamDataset(base_path / "validation.csv", max_length=max_length, tokenizer=tokenizer)
test_dataset = SpamDataset(base_path / "test.csv", max_length=max_length, tokenizer=tokenizer)
```

或者，我们可以通过以下方式定义 `max_length`：

```python
max_length = model.pos_emb.weight.shape[0]
```

或

```python
max_length = BASE_CONFIG["context_length"]
```

为方便起见，你可以通过以下命令运行此实验：

```bash
python additional-experiments.py --context_length "model_context_length"
```

使用 [../02_bonus_additional-experiments](../02_bonus_additional-experiments) 文件夹中的代码，结果测试准确率为 78.33%（而主章节中为 95.67%）。

> **结果分析**：将上下文长度从默认扩展到 1024 后，准确率从 95.67% 降至 78.33%。说明对于短文本分类，大量填充反而引入了噪声。

## 练习 6.2：微调整个模型

**练习目的**：不仅微调最后一个 Transformer 块，而是微调整个模型。

不是仅微调最后的 transformer 块，我们可以通过移除以下代码来微调整个模型：

```python
for param in model.parameters():
    param.requires_grad = False
```

为方便起见，你可以通过以下命令运行此实验：

```bash
python additional-experiments.py --trainable_layers all
```

使用 [../02_bonus_additional-experiments](../02_bonus_additional-experiments) 文件夹中的代码，测试准确率提升了 1%，达到 96.67%（而主章节中为 95.67%）。

> **结果分析**：微调整个模型比仅微调最后一层提升了 1%，但训练成本更高（所有参数都需要计算梯度）。

## 练习 6.3：微调第一个 vs 最后一个 token

**练习目的**：比较使用第一个 token vs 最后一个 token 进行分类的效果。

不是微调最后一个输出 token，我们可以微调第一个输出 token，将：

```python
model(input_batch)[:, -1, :]
```

改为：

```python
model(input_batch)[:, 0, :]
```

在代码中全部替换。

为方便起见，你可以通过以下命令运行此实验：

```
python additional-experiments.py --trainable_token first
```

使用 [../02_bonus_additional-experiments](../02_bonus_additional-experiments) 文件夹中的代码，测试准确率严重下降为 75.00%（而主章节中为 95.67%）。

> **结果分析**：最后一个 token 搼带了整个序列的上下文信息（因果注意力导致信息向右累积），而第一个 token 几乎没有上下文信息。因此用第一个 token 做分类效果很差。

```
三个练习对比总结:
  练习 6.1  context_length=1024  → 78.33% (填充太多噪声)
  练临 6.2  trainable_layers=all   → 96.67% (全量微调略优)
  练习 6.3  trainable_token=first  → 75.00% (第一个token缺乏上下文)
  主章节基线                    → 95.67%
```